# Tutorial 8: Using Prefab Pipelines

Building a pipeline from scratch gives you full control, but PhenoTypic
also ships **prefab pipelines** — pre-configured `ImagePipeline` subclasses
tuned for common organisms and plate types. In this tutorial you will
survey the available prefabs, apply one, and compare results.

**What you will learn:**

1. What prefab pipelines are and when to use them
2. Survey the available prefabs
3. Apply a prefab pipeline
4. Compare results from different prefabs

## Imports

In [1]:
from phenotypic.data import load_synth_yeast_plate

## Available Prefab Pipelines

Each prefab is an `ImagePipeline` subclass with operations and measurements
already configured. Choose based on your organism and plate conditions.

| Prefab | Best for | Strategy |
|--------|----------|----------|
| `HeavyOtsuPipeline` | General-purpose yeast, clean plates | Multi-stage Otsu with refinement |
| `HeavyWatershedPipeline` | Touching or clustered colonies | Watershed segmentation |
| `RoundPeaksPipeline` | Round colonies, lightweight | Peak detection |
| `HeavyRoundPeaksPipeline` | Round colonies needing refinement | Extended peak detection |
| `FilamentousFungiPipeline` | Filamentous fungi (*Neurospora*, etc.) | BM3D denoise + specialized detector |
| `GridSectionPipeline` | Pre-tiled grid sections | Section-level processing |

## Apply HeavyOtsuPipeline

Let's start with the most general-purpose option — `HeavyOtsuPipeline`.
It chains Gaussian blur, CLAHE, median filtering, Sobel edge enhancement,
Otsu detection, and several refinement steps (morphological opening,
border removal, small-object removal, mask fill).

In [2]:
from phenotypic.prefab import HeavyOtsuPipeline

plate = load_synth_yeast_plate()
heavy_otsu = HeavyOtsuPipeline()
result_otsu = heavy_otsu.apply(plate)
result_otsu.dash(overlay=True)

PNG file does not have exif data.


In [3]:
print(f"HeavyOtsuPipeline detected {result_otsu.num_objects} colonies")

HeavyOtsuPipeline detected 95 colonies


## Apply RoundPeaksPipeline

Now let's try `RoundPeaksPipeline` — a lighter approach that uses peak
detection to find circular colonies. It is faster but may miss irregular
or faint colonies.

In [4]:
from phenotypic.prefab import RoundPeaksPipeline

plate2 = load_synth_yeast_plate()
round_peaks = RoundPeaksPipeline()
result_rp = round_peaks.apply(plate2)
result_rp.dash(overlay=True)

PNG file does not have exif data.


In [5]:
print(f"RoundPeaksPipeline detected {result_rp.num_objects} colonies")

RoundPeaksPipeline detected 96 colonies


## Compare

Different prefabs produce different results on the same plate. The best
choice depends on your colonies, your imaging conditions, and what you
need to measure. A quick visual comparison and colony count helps you
decide.

In [6]:
print(f"HeavyOtsuPipeline:   {result_otsu.num_objects} colonies")
print(f"RoundPeaksPipeline:  {result_rp.num_objects} colonies")

HeavyOtsuPipeline:   95 colonies
RoundPeaksPipeline:  96 colonies


## Prefabs Include Measurements

Prefab pipelines come with measurements pre-configured, so you can call
`.apply_and_measure()` directly — no need to add your own `meas` list.

In [7]:
plate3 = load_synth_yeast_plate()
df = heavy_otsu.apply_and_measure(plate3)
print(f"{len(df)} colonies measured across {df.shape[1]} features")
df.head()

PNG file does not have exif data.


95 colonies measured across 128 features


,Metadata_FileSuffix,Metadata_BitDepth,Metadata_ImageType,Metadata_ImageName,Object_Label,Bbox_CenterRR,Bbox_CenterCC,Bbox_MinRR,Bbox_MinCC,Bbox_MaxRR,...,Intensity_MaximumIntensity,Intensity_MeanIntensity,Intensity_MedianIntensity,Intensity_StandardDeviationIntensity,Intensity_CoefficientVarianceIntensity,Intensity_LowerQuartileIntensity,Intensity_UpperQuartileIntensity,Intensity_InterquartileRangeIntensity,Intensity_Density,Intensity_ConvexDensity
0,.png,8,GridImage,Synthetic96PlateWithObjects,1,65.652950,675.196650,36,647,96,...,0.943073,0.728151,0.634163,0.175066,0.240513,0.556871,0.922481,0.365611,0.728151,10.812131
1,.png,8,GridImage,Synthetic96PlateWithObjects,2,67.201498,122.539230,38,93,98,...,0.931309,0.722491,0.622116,0.174493,0.241602,0.552666,0.915622,0.362956,0.722491,10.842557
2,.png,8,GridImage,Synthetic96PlateWithObjects,3,63.873453,248.427785,38,223,91,...,0.932142,0.711096,0.573390,0.170858,0.240385,0.560792,0.919544,0.358752,0.711096,9.426150
3,.png,8,GridImage,Synthetic96PlateWithObjects,4,64.969736,554.721141,38,528,92,...,0.939152,0.712699,0.573390,0.172119,0.241607,0.557704,0.919544,0.361840,0.712699,9.720417
4,.png,8,GridImage,Synthetic96PlateWithObjects,5,66.824299,60.551028,39,32,97,...,0.927387,0.734639,0.734702,0.169699,0.231083,0.556871,0.910585,0.353714,0.734639,10.755139


## Summary

Prefab pipelines are the fastest path from plate image to results:

- **`HeavyOtsuPipeline`** — robust general-purpose detection with refinement
- **`RoundPeaksPipeline`** — lightweight peak-based detection for round colonies
- **`HeavyWatershedPipeline`** — for touching or clustered colonies
- **`FilamentousFungiPipeline`** — specialized for branching fungal morphology
- All prefabs support `.apply()`, `.apply_and_measure()`, `.to_json()`, etc.

Choose based on your organism, plate conditions, and desired accuracy.
When a prefab is close but not quite right, use it as a starting point
and customize the parameters.

**Next up:** [Tutorial 9: Diagnosing Image Quality](09_diagnosing_image_quality.ipynb) —
assess plate quality before choosing a pipeline.